In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
# Load data
df = pd.read_csv('IMDB Dataset.csv')
x = df['review'].values
y = df['sentiment'].map({'positive': 1, 'negative': 0}).values

# Split before fitting the tokenizer
xtr, xte, ytr, yte = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Learn vocabulary from training reviews
tokenizer = Tokenizer(num_words=1000, oov_token='<OOV>')
tokenizer.fit_on_texts(xtr)

# Convert reviews to padded sequences
xtr = pad_sequences(tokenizer.texts_to_sequences(xtr), maxlen=200)
xte = pad_sequences(tokenizer.texts_to_sequences(xte), maxlen=200)

In [ ]:
# Build, train and evaluate both models
histories = {}

for name, layer in [('LSTM', LSTM), ('GRU', GRU)]:

    model = Sequential([
        Input(shape=(200,)),
        Embedding(1000, 100),
        layer(64),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        xtr, ytr,
        epochs=5,
        batch_size=32,
        validation_split=0.2
    )

    histories[name] = history

    loss, acc = model.evaluate(xte, yte)
    print(name, 'Accuracy:', round(acc * 100, 2))

In [ ]:
# Compare validation accuracy and loss
for metric in ['val_accuracy', 'val_loss']:
    plt.figure()

    for name, h in histories.items():
        plt.plot(h.history[metric], label=name)

    plt.xlabel('Epochs')
    plt.ylabel(metric)
    plt.title(metric)
    plt.legend()
    plt.show()